# LAB 08 — Support Vector Machines
## Kernels, Margins, and Model Selection on Real Customer-Churn Data

## Student Information
- **Name:** Sania
- **Student ID:** 023-24-0277
- **Section:** H
- **GitHub:** https://github.com/saniamirani
- **Kaggle:** https://www.kaggle.com/saniam58
- **Dataset:** Telco Customer Churn (Kaggle)
- **Dataset Source:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

---

### Before You Start
- You already studied SVM theory in class (margin, support vectors, kernel trick, C, gamma). This notebook does **not** re-teach that theory.
- This is **not** a "load data → `SVC().fit()` → done" lab. Most marks come from experimentation, visualization and justification, not from the fact that a model trained.
- You already know Python, Pandas, Matplotlib, `train_test_split`, encoding and evaluation metrics from Labs 2-6.
- Each task gives an **objective** and a **question** — not a code recipe. Hints tell you which tool is relevant; you decide how to use it.
- Do **not** provide complete code for every question yourself either — work it out. Every plot needs a written interpretation directly below it.


---

## Part 0 — Recap and Setup

Reload the Telco Customer Churn dataset and reapply the **same** cleaning/encoding pipeline from Labs 2/3/4/6:
- Convert `TotalCharges` to numeric and handle missing values
- Drop `customerID`
- Map `Churn` to 0/1
- One-hot encode categorical columns (`drop_first=True`)
- `train_test_split` with `stratify=y` — use the **same `random_state`** you used in Labs 3/4/6 so results stay comparable


In [1]:
# Load and clean the dataset (reuse your Lab 2/3 pipeline)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
from sklearn.model_selection import train_test_split

df = pd.read_csv("clean_churn.csv")
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df = df.dropna().copy()
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# One-hot encode, define X and y, train_test_split(..., stratify=y)
from sklearn.model_selection import train_test_split

y = df["Churn"].map({"Yes": 1, "No": 0}) 
X = df.drop(columns=["Churn"])
X = pd.get_dummies(X, drop_first=True, dtype=int)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


---

## Part 1 — Why Scaling Matters for SVM

**Task 1.1 — Objective:** Demonstrate what happens to an SVM when features are not scaled.

**What to do:**
1. Train a linear-kernel SVM (`SVC(kernel='linear')`) directly on your **un-scaled** encoded features.
2. Train a second linear-kernel SVM on the same split after applying `StandardScaler` (fit on train only, transform on train and test).
3. Compare accuracy, F1-score, training time, and any convergence warnings between the two runs.

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# unscaled
svm_raw = SVC(kernel='linear', random_state=42)
svm_raw.fit(X_train, y_train)

# scaled  (fit_transform on TRAIN, transform ONLY on TEST)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
```


In [ ]:
# Unscaled linear SVM
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from time import perf_counter

svm_raw = SVC(kernel="linear", random_state=42)
t0 = perf_counter()
svm_raw.fit(X_train, y_train)
raw_time = perf_counter() - t0
y_test_pred_raw = svm_raw.predict(X_test)
raw_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred_raw),
    "Precision": precision_score(y_test, y_test_pred_raw, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred_raw, zero_division=0),
    "F1-score": f1_score(y_test, y_test_pred_raw, zero_division=0),
}
print("Unscaled linear SVM")
print(pd.Series(raw_metrics).round(4))
print(f"Training time: {raw_time:.2f} seconds")

In [ ]:
# Scaled linear SVM (fit scaler on training data only)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svm_s = SVC(kernel="linear", random_state=42)
t0 = perf_counter()
svm_s.fit(X_train_s, y_train)
scaled_time = perf_counter() - t0
y_test_pred_s = svm_s.predict(X_test_s)
scaled_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred_s),
    "Precision": precision_score(y_test, y_test_pred_s, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred_s, zero_division=0),
    "F1-score": f1_score(y_test, y_test_pred_s, zero_division=0),
}
print("Scaled linear SVM")
print(pd.Series(scaled_metrics).round(4))
print(f"Training time: {scaled_time:.2f} seconds")

**Questions to answer**

**(a)** Compare the printed metric tables and training times above. Scaling may change accuracy and F1 because SVM depends on distances and dot products; report the actual values from your run rather than assuming scaling always improves every metric.

**(b)** Without scaling, features with larger numeric ranges can dominate the margin and decision boundary. Standardization puts features on comparable scales so the SVM can balance their influence.

**(c)** I use the scaled version for the remaining SVM experiments because scaling makes distance-based comparisons more meaningful. The scaler was fitted on training data only to avoid data leakage.

---

## Part 2 — Kernel Comparison

**Task 2.1 — Objective:** Train and fairly compare SVM classifiers using the linear, RBF and polynomial kernels on your **scaled** features.

**What to do:**
1. Train three `SVC` models — `kernel='linear'`, `kernel='rbf'`, `kernel='poly'` — with the same `random_state`, on the same scaled train split.
2. For each, record: Accuracy, Precision, Recall, F1-score, and number of support vectors (`model.support_.shape[0]`).
3. Assemble a single comparison table (a DataFrame is fine).
4. Plot the confusion matrix for whichever kernel performed best on F1-score.

| Syntax | Description |
|---|---|
| `SVC(kernel=...)` | `'linear'`, `'rbf'`, or `'poly'` |
| `model.support_` | indices of support vectors |
| `model.n_support_` | number of support vectors per class |


In [ ]:
# Train linear / RBF / polynomial SVMs and collect metrics
kernel_models = {}
kernel_rows = []
for kernel in ["linear", "rbf", "poly"]:
    model = SVC(kernel=kernel, random_state=42)
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    kernel_models[kernel] = model
    kernel_rows.append({
        "Kernel": kernel,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "Support vectors": model.support_.shape[0]
    })
kernel_results = pd.DataFrame(kernel_rows)
display(kernel_results.round(4))

In [ ]:
# Comparison table
display(kernel_results.sort_values("F1-score", ascending=False).reset_index(drop=True).round(4))
best_kernel_name = kernel_results.loc[kernel_results["F1-score"].idxmax(), "Kernel"]
best_kernel_model = kernel_models[best_kernel_name]
print("Best kernel by test F1-score:", best_kernel_name)

In [ ]:
# Confusion matrix for the best kernel
best_kernel_pred = best_kernel_model.predict(X_test_s)
ConfusionMatrixDisplay.from_predictions(
    y_test, best_kernel_pred, display_labels=["No churn", "Churn"], cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_kernel_name} SVM")
plt.tight_layout()
plt.show()

**Questions to answer**

**(a)** The kernel with the highest test F1-score is printed as `best_kernel_name` above. RBF can model nonlinear boundaries, while linear is simpler; the result depends on this dataset and split.

**(b)** Use the “Support vectors” column to identify the largest count. More support vectors can indicate that more training examples influence the boundary, but the count alone does not prove the boundary is more complex.

**(c)** Carry the RBF kernel into Part 4 if it gives the strongest F1-score or a useful precision/recall trade-off. Cite the exact F1 values from the comparison table.

---

## Part 3 — Visualizing the Margin: What Does C Actually Do?

**Task 3.1 — Objective:** Reduce your scaled features to 2 dimensions and visualize how an RBF-kernel SVM's decision boundary changes as C increases.

**What to do:**
1. Use `PCA(n_components=2)` on your scaled training features. Report the total explained variance ratio of the two components.
2. Train three RBF-kernel SVMs on this 2-D projection using a small, medium and large C (choose 3 values spanning at least two orders of magnitude, e.g. 0.01, 1, 100).
3. For each C, plot the 2-D decision boundary (mesh grid + `contourf`) with training points overlaid, and report the number of support vectors.

**Hint — Building the Mesh Grid:**
- Build a fine grid of (x, y) spanning slightly beyond your projected data's min/max using `np.meshgrid`.
- Call `model.predict()` on every grid point (flatten, then reshape back to grid shape).
- `plt.contourf(xx, yy, Z, alpha=0.3)` draws shaded regions; `plt.scatter(...)` overlays real points colored by true label.


In [ ]:
# PCA to 2 components, report explained variance ratio
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_s)
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 4))
print("Total variance explained by two components:", round(pca.explained_variance_ratio_.sum(), 4))

In [ ]:
# Train RBF SVMs at 3 different C values on the 2-D projection
C_values = [0.01, 1, 100]
pca_svm_models = {}
for c_value in C_values:
    model = SVC(kernel="rbf", C=c_value, gamma="scale", random_state=42)
    model.fit(X_train_2d, y_train)
    pca_svm_models[c_value] = model
    print(f"C={c_value}: {model.support_.shape[0]} support vectors")

In [ ]:
# Plot decision boundary for each C (mesh grid + contourf)
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, c_value in zip(axes, C_values):
    model = pca_svm_models[c_value]
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5, 0.5, 1.5])
    ax.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, s=9, alpha=0.55, cmap="coolwarm")
    ax.set_title(f"RBF SVM, C={c_value}\nSupport vectors: {model.support_.shape[0]}")
    ax.set_xlabel("Principal component 1")
    ax.set_ylabel("Principal component 2")
plt.tight_layout()
plt.show()

**Questions to answer**

**(a)** As C increases, misclassifying training examples is penalized more strongly, so the boundary generally fits the training data more closely and may become less smooth.

**(b)** The exact support-vector counts are printed for each C. Their trend is data-dependent; higher C does not guarantee that the count always increases or decreases.

**(c)** Judge underfitting by a boundary that is too simple to separate visible patterns; possible overfitting is suggested by a highly irregular boundary following local details. These are only visual clues, not proof of generalization performance.

**(d)** PCA projects the original scaled features into only two components, losing information. The plotted boundary is trained in this 2-D projection, so it is not the boundary of the full-feature SVM.

---

## Part 4 — Systematic Hyperparameter Search

**Task 4.1 — Objective:** Run `GridSearchCV` over a C x gamma grid for an RBF-kernel SVM and select the best configuration using cross-validated F1-score.

**What to do:**
1. Define a parameter grid with at least 3 values of C and at least 3 values of gamma.
2. Run `GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='f1')` on your scaled **training** data.
3. Extract `gs.cv_results_` and reshape the mean test scores into a C x gamma grid.
4. Visualize as a heatmap (`imshow` or a pivoted DataFrame) with C and gamma labeled.
5. Refit the best estimator and report its performance on the held-out **test** set.

| Syntax | Description |
|---|---|
| `GridSearchCV(estimator, param_grid, cv, scoring)` | tries every combination via k-fold CV |
| `gs.best_params_` | best hyperparameter combination |
| `gs.cv_results_['mean_test_score']` | mean CV score per combination |


In [ ]:
# Define param_grid and run GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, 0.1, 1]
}
gs = GridSearchCV(
    SVC(kernel="rbf", random_state=42),
    param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1, refit=True
)
gs.fit(X_train_s, y_train)
print("Best parameters:", gs.best_params_)
print("Best mean CV F1:", round(gs.best_score_, 4))

In [ ]:
# Heatmap of mean CV F1-score over the C x gamma grid
cv_results = pd.DataFrame(gs.cv_results_)
heatmap_data = cv_results.pivot(index="param_C", columns="param_gamma", values="mean_test_score")
heatmap_data = heatmap_data.sort_index().sort_index(axis=1)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="viridis", origin="lower")
ax.set_xticks(range(len(heatmap_data.columns)), [str(x) for x in heatmap_data.columns])
ax.set_yticks(range(len(heatmap_data.index)), [str(x) for x in heatmap_data.index])
ax.set_xlabel("gamma")
ax.set_ylabel("C")
ax.set_title("GridSearchCV Mean Cross-Validated F1-score")
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(j, i, f"{heatmap_data.iloc[i, j]:.3f}", ha="center", va="center", color="white" if heatmap_data.iloc[i,j] < heatmap_data.values.mean() else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Mean CV F1-score")
plt.tight_layout()
plt.show()

In [ ]:
# Refit best estimator, evaluate on test set
best_svm = gs.best_estimator_
best_svm_pred = best_svm.predict(X_test_s)
tuned_metrics = {
    "Accuracy": accuracy_score(y_test, best_svm_pred),
    "Precision": precision_score(y_test, best_svm_pred, zero_division=0),
    "Recall": recall_score(y_test, best_svm_pred, zero_division=0),
    "F1-score": f1_score(y_test, best_svm_pred, zero_division=0),
}
print("Tuned RBF SVM test metrics:")
display(pd.Series(tuned_metrics).round(4))
print("Untuned RBF F1:", round(kernel_results.loc[kernel_results.Kernel == "rbf", "F1-score"].iloc[0], 4))

**Questions to answer**

**(a)** The best C, gamma, and mean cross-validated F1-score are printed by `gs.best_params_` and `gs.best_score_`.

**(b)** Read the heatmap: a cluster of similarly high scores suggests a stable region; a single isolated high cell may indicate a more sensitive configuration.

**(c)** Compare `gs.best_score_` with the tuned test F1-score printed in Part 4. A large gap can indicate overfitting during model selection or that cross-validation and test data differ.

**(d)** Compare the tuned test F1-score with the untuned RBF F1 printed below the test metrics. Tuning is useful if the improvement is meaningful relative to the added search cost.

---

## Part 5 — Class Imbalance and the Business Trade-off

**Task 5.1 — Objective:** Compare your best RBF-kernel SVM with and without `class_weight='balanced'`, and reason about which is more useful for the business.

**What to do:**
1. Using your best (C, gamma) from Part 4, train two RBF SVMs: `class_weight=None` and `class_weight='balanced'`.
2. Report Precision, Recall, F1-score and confusion matrix for both on the test set.
3. Plot an ROC curve or Precision-Recall curve for the balanced model.


In [ ]:
# class_weight=None vs 'balanced' comparison
svm_plain = SVC(kernel="rbf", C=gs.best_params_["C"], gamma=gs.best_params_["gamma"],
                class_weight=None, probability=True, random_state=42)
svm_balanced = SVC(kernel="rbf", C=gs.best_params_["C"], gamma=gs.best_params_["gamma"],
                   class_weight="balanced", probability=True, random_state=42)
svm_plain.fit(X_train_s, y_train)
svm_balanced.fit(X_train_s, y_train)
plain_pred = svm_plain.predict(X_test_s)
balanced_pred = svm_balanced.predict(X_test_s)

def metric_row(name, model, pred):
    return {
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "TN": confusion_matrix(y_test, pred, labels=[0,1])[0,0],
        "FP": confusion_matrix(y_test, pred, labels=[0,1])[0,1],
        "FN": confusion_matrix(y_test, pred, labels=[0,1])[1,0],
        "TP": confusion_matrix(y_test, pred, labels=[0,1])[1,1],
    }
imbalance_results = pd.DataFrame([
    metric_row("Unweighted", svm_plain, plain_pred),
    metric_row("Balanced", svm_balanced, balanced_pred)
])
display(imbalance_results.round(4))
print("Unweighted confusion matrix:\n", confusion_matrix(y_test, plain_pred))
print("Balanced confusion matrix:\n", confusion_matrix(y_test, balanced_pred))

In [ ]:
# Precision-Recall curve for the balanced model
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

balanced_scores = svm_balanced.predict_proba(X_test_s)[:, 1]
PrecisionRecallDisplay.from_predictions(y_test, balanced_scores)
plt.title(f"Balanced RBF SVM — Precision-Recall (AP={average_precision_score(y_test, balanced_scores):.3f})")
plt.tight_layout()
plt.show()

**Questions to answer**

**(a)** Subtract the unweighted precision/recall values from the balanced values in the Part 5 table. Class weighting often increases churn recall while reducing precision, but use the observed numbers.

**(b)** Missing a genuine churner can mean losing a customer, while a false positive can waste retention-offer costs. The preferred model depends on the relative business costs; use the confusion-matrix counts to justify the choice.

**(c)** `probability=True` was enabled in `SVC()` to obtain probability estimates with `predict_proba()`; this adds training overhead and the values should not be treated as perfectly calibrated without checking calibration.

---

## Part 6 — Model Showdown: SVM vs. Everything Else

**Task 6.1 — Objective:** Build a single comparison table across all five algorithms (Decision Tree, Random Forest, Logistic Regression, KNN, SVM) and select ONE final model for production.

**What to do:**
1. Build a table: one row per model, columns Accuracy / Precision / Recall / F1-score. Re-run earlier models here on the same split if you don't have saved numbers.
2. Produce ONE bar chart comparing F1-score across all five models.


In [ ]:
# Cross-model comparison table (same train/test split)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

comparison_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Tuned RBF SVM": best_svm
}
comparison_rows = []
fitted_comparison_models = {}
for name, model in comparison_models.items():
    # SVM and KNN use scaled features; tree models can use raw encoded features.
    if name in ["Tuned RBF SVM", "KNN"]:
        model.fit(X_train_s, y_train)
        pred = model.predict(X_test_s)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    fitted_comparison_models[name] = model
    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0)
    })
model_comparison = pd.DataFrame(comparison_rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(model_comparison.round(4))
print("Best F1 model:", model_comparison.loc[model_comparison["F1-score"].idxmax(), "Model"])
print("Best recall model:", model_comparison.loc[model_comparison["Recall"].idxmax(), "Model"])

In [ ]:
# Bar chart of F1-score across models
plt.figure(figsize=(9, 5))
plt.bar(model_comparison["Model"], model_comparison["F1-score"])
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.title("Model Comparison — Test F1-score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

**Questions to answer**

**(a)** The Part 6 output names the highest-F1 and highest-recall models. They may be different because F1 balances precision and recall, while recall measures only the proportion of actual churners detected.

**(b)** The extra SVM training cost is worthwhile only if its test metrics offer a meaningful benefit for the business. Compare its F1/recall with the faster tree-based models.

**(c)** Recommend the model that best matches the business objective, referencing at least two metrics (for example, F1 and recall) and considering false-negative versus false-positive costs. Use the table's actual results.

---

## Part 7 — Final Predictions and Wrap-Up

**Task 7.1 — Objective:** Generate a predictions file and write a short executive summary.

**What to do:**
1. Using your final chosen model, generate predictions on the test set and save `svm_predictions.csv` with true label, predicted label, and predicted churn probability/score.
2. Write a 5-8 sentence Executive Summary below: which model you chose, key metrics, the main trade-off you made in Part 5, and one limitation of your analysis.


In [ ]:
# Generate and save svm_predictions.csv using the tuned RBF SVM
final_model = best_svm
final_predictions = final_model.predict(X_test_s)
final_scores = final_model.predict_proba(X_test_s)[:, 1]
predictions_df = pd.DataFrame({
    "true_churn": np.asarray(y_test),
    "predicted_churn": final_predictions,
    "predicted_churn_probability": final_scores
}, index=y_test.index)
predictions_df.to_csv("svm_predictions.csv", index=False)
print("Saved svm_predictions.csv")
display(predictions_df.head())

**Executive Summary**

This lab compared linear, RBF, and polynomial SVM kernels after standardizing features, then used PCA to visualize how C affects a boundary. GridSearchCV selected an RBF SVM configuration using five-fold cross-validated F1-score. The held-out test metrics and best hyperparameters are reported in Parts 4–6; insert the exact values from your run here. Class weighting was compared to examine the trade-off between churn recall and precision. The final model should be selected based on the cost of missed churners versus unnecessary retention offers, not accuracy alone. One limitation is that evaluation uses one train/test split and a single dataset, so performance may differ on future customers.

**Final Reflection**

**(a)** A defensible answer is the difference, if any, between cross-validation performance and held-out test performance, or the change in recall after class weighting. State the actual observation from your output.

**(b)** I would test threshold tuning and probability calibration, evaluate with repeated stratified cross-validation, and estimate business costs for false positives and false negatives.